In [5]:
# URL de repositorio
!git clone https://github.com/ZagonTM/Sen4Sam
# Una vez clonado, nos movemos al directorio del repositorio o lo añadimos al path
import os
if os.path.exists('Sen4Sam'):
    %cd Sen4Sam
    print("Acceso al repositorio Sen4Sam exitoso.")
else:
    print("No se pudo encontrar la carpeta del repositorio. Verifica la URL de clonación.")

# Instalar Segment Anything 2 (SAM 2) y sus dependencias
!pip install -q git+https://github.com/facebookresearch/sam2.git

# Crear directorio de checkpoints y descargar los pesos del modelo
!mkdir -p checkpoints
!wget -P checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt

Cloning into 'Sen4Sam'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 140 (delta 60), reused 106 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 3.14 MiB | 20.88 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/Sen4Sam/Sen4Sam
Acceso al repositorio Sen4Sam exitoso.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
--2026-11-08 16:30:11--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 54.230.79.122, 54.230.79.88, 54.230.79.53, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|54.230.79.122|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 184416285 (176M) [application/vnd.snesdev-page-table]
Saving to: ‘checkpoints/sam2.1_hiera_small.

In [15]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image

from google.colab import drive

ROOT = Path.cwd()

DATA_ROOT = Path(
    "/content/drive/MyDrive/Sen4SAM"
)

PROCESSED = (
    DATA_ROOT /
    "processed" /
    "processed"
)

sys.path.insert(
    0,
    str(ROOT / "src")
)
from sam2_auto_segment import build_model

model = build_model()

## Experimento 1: Exploración de la arquitectura del decoder
Objetivo:

Identificar los módulos internos del decoder de SAM2 y localizar posibles puntos de intervención para la adaptación a imágenes Sentinel‑2.

Hipótesis:

Si el problema de cambio de dominio ocurre dentro del decoder, entonces debería existir una ruta interna diferenciable para generar máscaras sin depender de la API de inferencia del predictor.




In [16]:
print(type(model.sam_mask_decoder))

<class 'sam2.modeling.sam.mask_decoder.MaskDecoder'>


In [17]:
import inspect

print(
    dir(model.sam_mask_decoder)
)

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_dynamic_multimask_via_stability', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_get_stability_scores', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_mod

In [18]:
for name in dir(model.sam_mask_decoder):

    if "forward" in name.lower():

        print(name)

_forward_hooks
_forward_hooks_always_called
_forward_hooks_with_kwargs
_forward_pre_hooks
_forward_pre_hooks_with_kwargs
_slow_forward
forward
register_forward_hook
register_forward_pre_hook


In [19]:
for name in dir(model.sam_mask_decoder):

    if "predict" in name.lower():

        print(name)

iou_prediction_head
predict_masks


In [20]:
import inspect

print(
    inspect.signature(
        model.sam_mask_decoder.forward
    )
)

(image_embeddings: torch.Tensor, image_pe: torch.Tensor, sparse_prompt_embeddings: torch.Tensor, dense_prompt_embeddings: torch.Tensor, multimask_output: bool, repeat_image: bool, high_res_features: Optional[List[torch.Tensor]] = None) -> Tuple[torch.Tensor, torch.Tensor]


In [21]:
print(
    inspect.getsource(
        model.sam_mask_decoder.forward
    )
)

    def forward(
        self,
        image_embeddings: torch.Tensor,
        image_pe: torch.Tensor,
        sparse_prompt_embeddings: torch.Tensor,
        dense_prompt_embeddings: torch.Tensor,
        multimask_output: bool,
        repeat_image: bool,
        high_res_features: Optional[List[torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Predict masks given image and prompt embeddings.

        Arguments:
          image_embeddings (torch.Tensor): the embeddings from the image encoder
          image_pe (torch.Tensor): positional encoding with the shape of image_embeddings
          sparse_prompt_embeddings (torch.Tensor): the embeddings of the points and boxes
          dense_prompt_embeddings (torch.Tensor): the embeddings of the mask inputs
          multimask_output (bool): Whether to return multiple masks or a single
            mask.

        Returns:
          torch.Tensor: batched predicted masks
          torch.Tensor: batched pred

In [22]:
dir(model.sam_mask_decoder)

['T_destination',
 '__annotations__',
 '__call__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_apply',
 '_backward_hooks',
 '_backward_pre_hooks',
 '_buffers',
 '_call_impl',
 '_compiled_call_impl',
 '_dynamic_multimask_via_stability',
 '_forward_hooks',
 '_forward_hooks_always_called',
 '_forward_hooks_with_kwargs',
 '_forward_pre_hooks',
 '_forward_pre_hooks_with_kwargs',
 '_get_backward_hooks',
 '_get_backward_pre_hooks',
 '_get_name',
 '_get_stability_scores',
 '_is_full_backward_hook',
 '_load_from_state_dict',
 '_load_state_dict_post_hooks',
 '_load_state_dict_pr

In [23]:
import inspect

print(
    inspect.signature(
        model.sam_mask_decoder.predict_masks
    )
)

(image_embeddings: torch.Tensor, image_pe: torch.Tensor, sparse_prompt_embeddings: torch.Tensor, dense_prompt_embeddings: torch.Tensor, repeat_image: bool, high_res_features: Optional[List[torch.Tensor]] = None) -> Tuple[torch.Tensor, torch.Tensor]


In [24]:
import inspect

print(
    inspect.signature(
        model.sam_mask_decoder.forward
    )
)

(image_embeddings: torch.Tensor, image_pe: torch.Tensor, sparse_prompt_embeddings: torch.Tensor, dense_prompt_embeddings: torch.Tensor, multimask_output: bool, repeat_image: bool, high_res_features: Optional[List[torch.Tensor]] = None) -> Tuple[torch.Tensor, torch.Tensor]


In [25]:
src = inspect.getsource(
    model.sam_mask_decoder.predict_masks
)

print(src[:5000])

    def predict_masks(
        self,
        image_embeddings: torch.Tensor,
        image_pe: torch.Tensor,
        sparse_prompt_embeddings: torch.Tensor,
        dense_prompt_embeddings: torch.Tensor,
        repeat_image: bool,
        high_res_features: Optional[List[torch.Tensor]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Predicts masks. See 'forward' for more details."""
        # Concatenate output tokens
        s = 0
        if self.pred_obj_scores:
            output_tokens = torch.cat(
                [
                    self.obj_score_token.weight,
                    self.iou_token.weight,
                    self.mask_tokens.weight,
                ],
                dim=0,
            )
            s = 1
        else:
            output_tokens = torch.cat(
                [self.iou_token.weight, self.mask_tokens.weight], dim=0
            )
        output_tokens = output_tokens.unsqueeze(0).expand(
            sparse_prompt_embeddings.size(0), -1,

### Resultado
Se identificó que el decoder posee dos funciones principales:

* forward()
* predict_masks()

Además, se confirmó que predict_masks() devuelve:

* masks
* iou_pred
* mask_tokens_out
* object_score_logits

lo que indica la existencia de una ruta diferenciable apta para entrenamiento.

##Experimento 2: Inspección del predictor
Objetivo:

Determinar qué información interna conserva SAM2ImagePredictor después de procesar una imagen.

Hipótesis:

El predictor debería almacenar las características generadas por el encoder para posteriormente alimentar al decoder.

In [29]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

predictor = SAM2ImagePredictor(model)

print(type(predictor))

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


<class 'sam2.sam2_image_predictor.SAM2ImagePredictor'>


In [40]:
def load_rgb(patch):

    return np.array(
        Image.open(
            DATA_ROOT / "processed"/ "processed" / f"{patch}_rgb.png"
        ).convert("RGB")
    )
def load_parcels(patch):

    return np.load(
        DATA_ROOT / "processed"/ "processed" / f"{patch}_parcels.npy"
    )

from google.colab import drive

drive.mount('/content/drive')

gt_dataset = pd.read_csv(
    DATA_ROOT / "gt_parcel_dataset.csv"
)

train_df = gt_dataset.sample(
    200,
    random_state=42
)


row = train_df.iloc[0]

image = load_rgb(
    row["patch"]
)

Mounted at /content/drive


In [41]:
predictor.set_image(image)

In [42]:
print(type(predictor._features))
print(predictor._features.keys())

<class 'dict'>
dict_keys(['image_embed', 'high_res_feats'])


In [33]:
print(predictor.__dict__.keys())

dict_keys(['model', '_transforms', '_is_image_set', '_features', '_orig_hw', '_is_batch', 'mask_threshold', '_bb_feat_sizes'])


In [31]:
for k, v in predictor.__dict__.items():

    if hasattr(v, "shape"):

        print(k, v.shape)

In [32]:
for k, v in predictor.__dict__.items():

    print(k, type(v))

model <class 'sam2.modeling.sam2_base.SAM2Base'>
_transforms <class 'sam2.utils.transforms.SAM2Transforms'>
_is_image_set <class 'bool'>
_features <class 'NoneType'>
_orig_hw <class 'NoneType'>
_is_batch <class 'bool'>
mask_threshold <class 'float'>
_bb_feat_sizes <class 'list'>


###Resultado
Tras ejecutar set_image(), el predictor almacena:

* image_embed
* high_res_feats

confirmando que las características del encoder permanecen accesibles para futuras llamadas al decoder.

##Experimento 3: Caracterización de embeddings
Objetivo:

Identificar la dimensionalidad y estructura de las representaciones visuales producidas por el encoder.

Hipótesis:

Las características extraídas deberían coincidir con las entradas requeridas por el decoder.

In [43]:
for k,v in predictor._features.items():

    print("\n", k)

    print(type(v))

    if hasattr(v,"shape"):

        print(v.shape)

    elif isinstance(v,list):

        print("list length:",len(v))

        for i,item in enumerate(v):

            print(i,item.shape)


 image_embed
<class 'torch.Tensor'>
torch.Size([1, 256, 64, 64])

 high_res_feats
<class 'list'>
list length: 2
0 torch.Size([1, 32, 256, 256])
1 torch.Size([1, 64, 128, 128])


In [44]:
print(dir(model.sam_prompt_encoder))

['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_embed_boxes', '_embed_masks', '_embed_points', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_batch_size', '_get_device', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full

### Resultado
Se identificaron:

image_embed:

* forma (1,256,64,64)

high_res_feats:

* (1,32,256,256)
* (1,64,128,128)

Estas representaciones coinciden con la estructura esperada por predict_masks().

##Experimento 4: Análisis del Prompt Encoder
Objetivo:

Determinar cómo se generan los embeddings de entrada para el decoder a partir de puntos, cajas o máscaras.

Hipótesis:

El Prompt Encoder debe producir tanto embeddings esparsos como embeddings densos para alimentar predict_masks().

In [45]:
import inspect

print(
    inspect.signature(
        model.sam_prompt_encoder.forward
    )
)

(points: Optional[Tuple[torch.Tensor, torch.Tensor]], boxes: Optional[torch.Tensor], masks: Optional[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]


In [46]:
src = inspect.getsource(
    model.sam_prompt_encoder.forward
)

print(src[:3000])

    def forward(
        self,
        points: Optional[Tuple[torch.Tensor, torch.Tensor]],
        boxes: Optional[torch.Tensor],
        masks: Optional[torch.Tensor],
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Embeds different types of prompts, returning both sparse and dense
        embeddings.

        Arguments:
          points (tuple(torch.Tensor, torch.Tensor) or none): point coordinates
            and labels to embed.
          boxes (torch.Tensor or none): boxes to embed
          masks (torch.Tensor or none): masks to embed

        Returns:
          torch.Tensor: sparse embeddings for the points and boxes, with shape
            BxNx(embed_dim), where N is determined by the number of input points
            and boxes.
          torch.Tensor: dense embeddings for the masks, in the shape
            Bx(embed_dim)x(embed_H)x(embed_W)
        """
        bs = self._get_batch_size(points, boxes, masks)
        sparse_embeddings = torch.empty(
           

### Resultado
Se confirmó que el Prompt Encoder recibe:

* points
* boxes
* masks

y devuelve:

* sparse_embeddings
* dense_embeddings

Estos tensores constituyen las entradas restantes necesarias para ejecutar el decoder de forma independiente.

#Conclusión General
Los experimentos realizados permitieron reconstruir la ruta interna completa utilizada por SAM2 para generar máscaras.

Se identificaron:

1. El decoder y sus funciones diferenciables.
2. Los embeddings visuales producidos por el encoder.
3. Las características multiescala utilizadas durante la generación de máscaras.
4. El mecanismo de codificación de prompts.

Como resultado, se dispone de todos los componentes necesarios para implementar un proceso de fine‑tuning supervisado del decoder utilizando parcelas agrícolas de Sen4AgriNet como verdad de referencia.

La siguiente etapa corresponde al Notebook 06, donde se construirá el pipeline de entrenamiento basado en:

Imagen RGB → Prompt Encoder → Mask Decoder → Dice Loss → Actualización de parámetros.